In [1]:
import sys
import os
parent_dir = os.path.abspath(os.path.join(".."))
sys.path.append(parent_dir)



import networkx as nx
import numpy as np
import torch
import torch_geometric as pyg
from py_modules.feature_engineering import get_ProNE_emb, get_node_centrality_nx, get_ones_attrs
import itertools
from torch_geometric.utils import from_networkx
from torch_geometric.data import Data, Batch
import os
import dgl

EXPORT_DIR = os.path.join('Data', 'SSL_data')
export_name = 'SSL_data_SW_BA_ER_100_500_noProcAttr'


os.makedirs(EXPORT_DIR, exist_ok=True)
node_attrs_dict = {
    "procedural_attrs": [
        # "degree",
        # "pagerank",
        "ones"
    ],
    "procedural_attrs_args": {
        # "degree": {
        #     "normalize": True
        # },
        # "pagerank": {
        #     "normalize": True
        # },
        "ones": {
            "num_features": 2
        }
    },
}


In [2]:
Gs = {
    'newman_watts_strogatz_graph': {
        'n': [100, 200, 300, 400, 500],
        'k': [4, 8, 12],
        'p': [0.1, 0.2, 0.3],
        'seed': [9, 20, 42, 3000, 4042, 4240, 4242, 20202, 101010, 202020]
    },
    'barabasi_albert_graph': {
        'n': [100, 200, 300, 400, 500],
        'm': [4, 8, 12],
        'seed': [9, 20, 42, 3000, 4042, 4240, 4242, 20202, 101010, 202020]
    },
    'erdos_renyi_graph': {
        'n': [100, 200, 300, 400, 500],
        'p': [0.01, 0.05, 0.1],
        'seed': [9, 20, 42, 3000, 4042, 4240, 4242, 20202, 101010, 202020]
    }
}


#count the number of graphs using itertools




In [3]:
def get_procedural_node_attributes(G_nx, node_attrs_dict, return_names=True):

    attributes = set(node_attrs_dict['procedural_attrs'])

    prone_emb = None
    if 'prone' in attributes:
        prone_args = node_attrs_dict['procedural_attrs_args'].get('prone', {})
        prone_emb, node_ids = get_ProNE_emb(G_nx, **prone_args)
        attributes.remove('prone')

    ones_np = None
    if 'ones' in attributes:
        ones_args = node_attrs_dict['procedural_attrs_args'].get('ones', {'num_features': 1})
        ones_np = get_ones_attrs(G_nx, **ones_args)
        attributes.remove('ones')
    
    attributes = list(attributes)
    cent_emb, cent_list_of_features = get_node_centrality_nx(G_nx, attributes, node_attrs_dict['procedural_attrs_args'])
    emb_names = cent_list_of_features

    if prone_emb is not None:
        cent_emb = np.concatenate((cent_emb, prone_emb), axis=1)
        prone_dim = prone_emb.shape[1]
        prone_names = [f'prone_{i}' for i in range(prone_dim)]
        emb_names += prone_names

    if ones_np is not None:
        if len(cent_emb) > 0:
            cent_emb = np.concatenate((cent_emb, ones_np), axis=1)
        else:
            cent_emb = ones_np
        ones_dim = ones_np.shape[1]
        ones_names = [f'ones_{i}' for i in range(ones_dim)]
        emb_names += ones_names


    cent_emb = cent_emb.astype(np.double)

    if return_names:
        return cent_emb, emb_names
    else:
        return cent_emb



In [4]:
def batch_to_single_data(batch):
    data = Data(
        x=batch.x,
        edge_index=batch.edge_index,
        edge_attr=batch.edge_attr if 'edge_attr' in batch else None,
        y=batch.y if 'y' in batch else None
    )
    return data


In [5]:

def generate_data_list(Gs_dict):
    data_list = []
    
    # Loop over each graph type and its parameter dictionary
    for graph_name, param_dict in Gs_dict.items():
        param_keys = list(param_dict.keys())
        param_values = list(param_dict.values())

        for combination in itertools.product(*param_values):

            # get the graph generator function
            graph_generator = getattr(nx, graph_name)
            param_set = dict(zip(param_keys, combination))

            # Build the NetworkX graph using the chosen parameters
            G = graph_generator(**param_set)

            # Convert the NetworkX graph to a PyG Data object
            data = from_networkx(G)

            node_attrs, node_attrs_names = get_procedural_node_attributes(G, node_attrs_dict, return_names=True)
            node_attrs = torch.tensor(node_attrs, dtype=torch.double)
            data.x = node_attrs

            # dgl_graph = dgl.from_networkx(G)
            # metis_partitions = dgl.metis_partition_assignment(dgl_graph, 5)
            # print(metis_partitions)

            # data.graph_name = graph_name
            # data.params = param_set

            data_list.append(data)

    return data_list


In [6]:
data_list = generate_data_list(Gs)

In [7]:
DATA_batch = Batch.from_data_list(data_list)

print(DATA_batch)
data_final = batch_to_single_data(DATA_batch)

DataBatch(edge_index=[2, 2876612], num_nodes=225000, x=[225000, 2], batch=[225000], ptr=[751])


In [8]:
# export the data
torch.save(data_final, os.path.join(EXPORT_DIR, f'{export_name}.pt'))

In [9]:
os.path.join(EXPORT_DIR, f'{export_name}.pt')

'Data/SSL_data/SSL_data_SW_BA_ER_100_500_noProcAttr.pt'